In [ ]:
# Reset env to erase previous vars, dicts, dfs, etc.
%reset -f

# Configuration & imports
import os
import pandas as pd
import numpy as np
from collections import defaultdict
from datetime import datetime
import re
from collections import OrderedDict
import openpyxl
import chardet

print("Environment reset. Configuration & imports done.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")


In [ ]:
# Specify CSV path
# Change this to point to the downloaded SPECIATE CSV
CSV_PATH = r"C:\Users\CountDracula\SPECIATE\SPECIATE_Profiles_All_02.csv"

# Quick existence check
assert os.path.exists(CSV_PATH), f"CSV file not found: {CSV_PATH}"
print("CSV path is valid.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")


In [ ]:
# Read CSV file
# Check encoding
with open(CSV_PATH, "rb") as f:
    result = chardet.detect(f.read(100000))
print(f"CSV file encoding: \n{result}\n")

print("Reading CSV with pandas.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
print("It will take a while...\n")

# Read entire CSV
df = pd.read_csv(CSV_PATH, encoding=result["encoding"], low_memory=False)

print("CSV loaded.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
print(f"\nCSV shape: \n{df.shape[0]:,} rows × {df.shape[1]} columns\n")


In [ ]:
# Check Name & CAS for odd values and edit the name
print("Checking 'Name' column for odd values.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

# Keep original names for diagnostics
df["Name_Original"] = df["Name"]
df["SPECIES_NAMES_Original"] = df["SPECIES_NAMES"]

df_rename = df.copy()

# Total number of rows
n_total = len(df_rename)

# Columns to clean
cols = ["CAS", "Name", "CAS no hyphen", "SPECIES_NAMES"]

for col in cols:
    # Create a boolean mask of rows containing one or more '»'
    mask = df_rename[col].str.contains(r"»+", regex=True, na=False)
    n_removed = mask.sum()
    # Remove all occurrences of '»' (one or more)
    df_rename[col] = df_rename[col].str.replace(r"»+", "", regex=True).str.strip()
    print(f"Column '{col}': '»' removed from {n_removed:,} rows")
print()


# Split Name on 1) ' ( or' and then 2) ' ||' (if still present) and keep first instance
df_rename["Name"] = (
    df_rename["Name"]
    .str.split(r"\s*\(or|\s*\|\|", n=1, regex=True)
    .str[0]
    .str.strip()
)

df_rename["SPECIES_NAMES"] = (
    df_rename["SPECIES_NAMES"]
    .str.split(r"\s*\(or|\s*\|\|", n=1, regex=True)
    .str[0]
    .str.strip()
)

# Names that contained patterns that could trigger stripping
mask_candidates = df_rename["Name_Original"].str.contains(r"\(or|\|\|", regex=True, na=False)
n_candidates = mask_candidates.sum()
mask_candidates_sp = df_rename["SPECIES_NAMES_Original"].str.contains(r"\(or|\|\|", regex=True, na=False)
n_candidates_sp = mask_candidates_sp.sum()

# Names that actually changed after cleaning
mask_changed = df_rename["Name_Original"] != df_rename["Name"]
n_changed = mask_changed.sum()
mask_changed_sp = df_rename["SPECIES_NAMES_Original"] != df_rename["SPECIES_NAMES"]
n_changed_sp = mask_changed_sp.sum()

print(f"Total names: \n{n_total:,}\n")
print(f"Names containing synonym patterns (to be fixed): \n{n_candidates:,}\n")
print(f"Names actually fixed: \n{n_changed:,}\n")
print(f"Total SPECIES names: \n{n_total:,}\n")
print(f"SPECIES names containing synonym patterns (to be fixed): \n{n_candidates_sp:,}\n")
print(f"SPECIES names actually fixed: \n{n_changed_sp:,}\n")

# Total number of rows before filtering
n_total = len(df_rename)

# Define mixture separators
separators = [";", "/", "+", "&"]

# Count occurrences of each separator
sep_counts = {}
for sep in separators:
    sep_counts[sep] = df_rename["Name"].str.contains(re.escape(sep), na=False).sum()

# Identify rows containing any mixture separator
pattern_any = r"|".join(map(re.escape, separators))
mask_mixtures = df_rename["Name"].str.contains(pattern_any, regex=True, na=False)
n_candidates = mask_mixtures.sum()

# Remove mixture rows
df_rename_clean = df_rename[~mask_mixtures].reset_index(drop=True)
n_removed = n_total - len(df_rename_clean)

# Print diagnostics
print(f"Total rows before Name filtration: \n{n_total:,}\n")
print(f"Names containing mixture patterns (to be removed): \n{n_candidates:,}\n")
print(f"Names actually removed: \n{n_removed:,}\n")
print(f"Remaining rows after filtering: \n{len(df_rename_clean):,}\n")
print("Number of names containing each separator pattern:")
for sep, count in sep_counts.items():
    print(f"  '{sep}': {count:,}")

# Count occurrences of each separator
sep_counts_cas = {}
for sep in separators:
    sep_counts_cas[sep] = df_rename_clean["CAS"].str.contains(re.escape(sep), na=False).sum()

# Identify rows containing any mixture separator
pattern_any_cas = r"|".join(map(re.escape, separators))
mask_mixtures_cas = df_rename_clean["CAS"].str.contains(pattern_any_cas, regex=True, na=False)
n_candidates_cas = mask_mixtures_cas.sum()

# Remove mixture rows
df_clean_cas = df_rename_clean[~mask_mixtures_cas].reset_index(drop=True)
n_removed_cas = len(df_rename_clean) - len(df_clean_cas)

# Print diagnostics
print(f"\nTotal rows before CAS# filtration: \n{len(df_rename_clean):,}\n")
print(f"CAS# containing mixture patterns (to be removed): \n{n_candidates_cas:,}\n")
print(f"CAS# actually removed: \n{n_removed_cas:,}\n")
print(f"Remaining rows after filtering: \n{len(df_clean_cas):,}\n")
print("Number of CAS# containing each separator pattern:")
for sep, count in sep_counts_cas.items():
    print(f"  '{sep}': {count:,}")    

print(f"\nFiltered DF shape: \n{df_clean_cas.shape[0]:,} rows × {df_clean_cas.shape[1]} columns\n")


In [ ]:
# Create weight_fraction column
print("Creating 'weight_fraction' column.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

df_move = df_clean_cas.copy()

wt_pc = df_move["WEIGHT_PERCENT"]

df_move["weight_fraction"] = (wt_pc / 100)

cols = list(df_move.columns)

cols.remove("weight_fraction")
wt_pc_idx = cols.index("WEIGHT_PERCENT")
cols.insert(wt_pc_idx, "weight_fraction")

cols.remove("Name_Original")
cols.insert(wt_pc_idx - 1, "Name_Original")

cols.remove("SPECIES_NAMES_Original")
cols.insert(wt_pc_idx + 6, "SPECIES_NAMES_Original")

df_wf = df_move[cols]

print("Created & populated 'weight_fraction' column.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")


In [ ]:
# Format CAS and ensure text format to avoid Excel conversion
print("Formatting CAS.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

def format_cas(cas):
    if pd.isna(cas):
        return None

    # convert to string and strip unwanted chars and whitespace
    cas_str = str(cas).strip().lstrip("»»").rstrip("»»").strip()

    # remove decimal part if present
    cas_str = cas_str.split(".")[0]

    if len(cas_str) < 4:
        return None

    # format as regular CAS number
    return f"{cas_str[:-3]}-{cas_str[-3:-1]}-{cas_str[-1]}"

df_wf["CAS#"] = df_wf["CAS no hyphen"].apply(format_cas).astype(str)

cols_df_wf = list(df_wf.columns)
cas_idx = cols_df_wf.index("CAS")
cols_df_wf.remove("CAS#")
cols_df_wf.insert(cas_idx + 1, "CAS#")
df_wf = df_wf[cols_df_wf]

print("Done formatting CAS#.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")
print(f"New CAS# column: \n{df_wf['CAS#']}\n")


In [ ]:
# Save unique Name & CAS# & DSSTox_ID set for PubChem data retrieval
print("Creating unique 'Name' & 'CAS#' & 'DSSTox_ID' set for PubChem data retrieval.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

# Select only Name and CAS# columns
df_unique = df_wf[["Name", "SPECIES_NAMES", "CAS#", "DSSTox_ID", "Smiles Notation"]]

# Drop duplicates to get unique Name-CAS pairs
df_unique = df_unique.drop_duplicates()

# Optional: reset index
df_unique = df_unique.reset_index(drop=True)

# Prevent Excel CAS# conversion
df_unique["CAS#"] = df_unique["CAS#"].astype(str)
# Save to CSV or Exceld
df_unique.to_excel(r"C:\Users\CountDracula\SPECIATE\SPECIATE_Profiles_All_03_CAS&Name&SMILES.xlsx", index=False)

print(f"Saved {len(df_unique)} unique Name-CAS-DSSToxID pairs. {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")


In [ ]:
# Write filtered SPECIATE to CSV file
print("Writing filtered SPECIATE to CSV.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

# Prevent Excel CAS# conversion
df_wf["CAS#"] = df_wf["CAS#"].astype(str)
# Save SPECIATE BEFORE further filtration
df_wf.to_csv(r"C:\Users\CoutDracula\SPECIATE\SPECIATE_Profiles_All_03_Fltrd.csv", index=False)

print("Filtered SPECIATE written to CSV.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

print(f"Filtered DF shape: \n{df_wf.shape[0]:,} rows × {df_wf.shape[1]} columns\n")


In [ ]:
# QC print
print(f"Filtered SPECIATE (intermediate) columns:")
for col in df_wf.columns:
    print(col)
print()


In [ ]:
# Specify path for SPECIATE updated with PubChem InChIKeys & other data
# Change this to point to the updated SPECIATE Excel file
XL_PATH = r"C:\Users\CountDracula\SPECIATE\SPECIATE_Profiles_All_04_Fltrd_RetrievedData.xlsx"

# Quick existence check
assert os.path.exists(XL_PATH), f"Excel file not found: {XL_PATH}"
print("Updated SPECIATE Excel path is valid.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")


In [ ]:
# Read updated SPECIATE Excel file
print("Reading updated SPECIATE Excel file.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
print("It may take a while for large files...\n")

# Read entire Excel file
df_xl = pd.read_excel(XL_PATH, sheet_name=0, engine="openpyxl")

print("Updated SPECIATE Excel loaded.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
print(f"\nUpdated SPECIATE Excel shape: \n{df_xl.shape[0]:,} rows × {df_xl.shape[1]} columns\n")


In [ ]:
# QC print
print(f"Updated SPECIATE columns:")
for col in df_xl.columns:
    print(col)
print()


In [ ]:
# Filter database by keywords
print("Filtering SPECIATE by InChIKey & other keywords.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

df_upd = df_xl.copy()

initial_n = len(df_upd)

print(f"Number of entries in SPECIATE before InChIKey & keyword filtration: \n{initial_n:,}\n")

# Keep individual chemicals only
# Remove entries with no InChIKey values (mixtures or unidentified features)
df_inchikey = df_upd[df_upd["InChIKey"].notna()].copy()

print(f"Dropped {(initial_n - len(df_inchikey)):,} entries with no InChIKey values. \n\nRemaining rows: \n{len(df_inchikey):,}\n")


# List of InChIKeys to drop (e.g., mixtures, inorganic compounds & inorganic mixtures, etc.)
drop_inchikeys = [
    "ATJFFYVFTNAWJD-UHFFFAOYSA-N", "NINIDFKCEFEMDL-UHFFFAOYSA-N", "AKEJUJNQAAGONA-UHFFFAOYSA-N", # Tin; Sulfur; Sulfur trioxide
    "QAOWNCQODCNURD-UHFFFAOYSA-N", "QHMQWEPBXSHHLH-UHFFFAOYSA-N", "PXJJSXABGXMUSU-UHFFFAOYSA-N", # Sulfuric Acid; Sulfur tetrafluoride; Sulfur monochloride
    "SFZCNBIFKDRMGX-UHFFFAOYSA-N", "RAHZWNYVWXNFOC-UHFFFAOYSA-N", "LSNNMFCWUKXFEE-UHFFFAOYSA-M", # Sulfur Hexafluoride; Sulfur Dioxide; Hydrogen sulfite
    "QAOWNCQODCNURD-UHFFFAOYSA-L", "GEHJYWRUCIMESM-UHFFFAOYSA-L", "KEAYESYHFKHZAL-UHFFFAOYSA-N", # Sulfate; Sodium Sulfite; Sodium
    "ZLMJMSJWJFRBEC-UHFFFAOYSA-N", "OAICVXFJPJFONN-UHFFFAOYSA-N", "NBIIXXVUZAFLBC-UHFFFAOYSA-N", # Potassium; Phosphorus; Phosphoric acid
    "ABLZXFCXXLZCGV-UHFFFAOYSA-N", "UEZVMMHDMIWARA-UHFFFAOYSA-M", "NBIIXXVUZAFLBC-UHFFFAOYSA-K", # Phosphonic acid; Phosphonate; Phosphate
    "IJGRMHOSHXDMSA-UHFFFAOYSA-N", "GKAOGPIIYCISHV-UHFFFAOYSA-N", "PNDPGZBMCMUPRI-UHFFFAOYSA-N", # Nitrogen; Neon; Iodine
    "UFHFLCQGNIYNRP-UHFFFAOYSA-N", "PXGOKWXKJXAPGV-UHFFFAOYSA-N", "KRHYYFGTRYWZRS-UHFFFAOYSA-M", # Hydrogen; Fluorine; Fluoride
    "XFXPMWWXUTWYJX-UHFFFAOYSA-N", "IQPQWNKOIGAROB-UHFFFAOYSA-N", "IXBPPZBJIFNGJJ-UHFFFAOYSA-N", # Cyanide; Cyanate; Cyanamide,cyano-,sodium salt
    "XZMCDFZZKTWFGF-UHFFFAOYSA-N", "VEXZGXHMUGYJMC-UHFFFAOYSA-M", "BVKZGUZCCUSVTD-UHFFFAOYSA-L", # Cyanamide; Chloride; Carbonate
    "VNWKTOKETHGBQD-NJFSPNSNSA-N", "OKTJSMMVPCPJKN-UHFFFAOYSA-N", "GDTBXPJZTBHREO-UHFFFAOYSA-N", # Carbon-14; Carbon; Bromine
    "CPELXLSAUQHCOX-UHFFFAOYSA-M", "UXNBTDLSBQFMEH-UHFFFAOYSA-N", "QGZKDVFQNNGYKY-UHFFFAOYSA-O", # Bromide; Brass; Ammonium
    "QGZKDVFQNNGYKY-UHFFFAOYSA-N", "QTBSBXVTEAMEQO-UHFFFAOYSA-M", "DVARTQFDIMZBAA-UHFFFAOYSA-O", # Ammonia; Acetate; Ammonium nitrate
    "NLXLAEXVIDQMFP-UHFFFAOYSA-N", "BFNBIHQBYMNNAN-UHFFFAOYSA-N", "XLYOFNOQVPJJNP-UHFFFAOYSA-N", # Ammonium chloride; Ammonium sulfate; Water
    "QMMFVYPAHWMCMS-UHFFFAOYSA-N", "HEDRZPFGACZZDS-UHFFFAOYSA-N", "XYFCBTPGUUZFHI-UHFFFAOYSA-N", # Dimethyl sulfide; Chloroform; Phosphine
    "AQSJGOWTSHOLKH-UHFFFAOYSA-N", "USFZMSVCRYTOJT-UHFFFAOYSA-N", "AJPJDKMHJJGVTQ-UHFFFAOYSA-K", # Phosphite; Ammonium acetate; Sodium phosphate
    "SWLVFNYSXGMGBS-UHFFFAOYSA-N", "DDSPUNTXKUFWTM-UHFFFAOYSA-N", "UKFWSNCTAHXBQN-UHFFFAOYSA-N", # Ammonium bromide; Tin oxide; Ammonium iodide
    "HEMHJVSKTPXQMS-UHFFFAOYSA-M", "VHUUQVKOLVNVRT-UHFFFAOYSA-N", "AVXURJPOCDRRFD-UHFFFAOYSA-N", # Sodium hydroxide; Ammonium hydroxide; Hydroxylamine
    "LDDQLRUQCUTJBB-UHFFFAOYSA-N", "QXNVGIXVLWOKEQ-UHFFFAOYSA-N", "BTIJJDXEELBZFS-UHFFFAOYSA-K", # Ammonium fluoride; Disodium; Hemin
    "GYYDPBCUIJTIBM-DYOGSRDZSA-N", "XLYOFNOQVPJJNP-UHFFFAOYSA-M", "NHNBFGGVMKEFGY-UHFFFAOYSA-N", # Agar; Hydroxide; Nitrate
    "BVKZGUZCCUSVTD-UHFFFAOYSA-N", "JYYOBHFYCIDXHH-UHFFFAOYSA-N", "NJWBAJNKROKYMV-UHFFFAOYSA-N", # Carbonic acid; Carbonic acid--water (1/1); C 15
    "NBIIXXVUZAFLBC-ZRLBSURWSA-N", "CWYNVVGOOAEACU-UHFFFAOYSA-N", "UVDDHYAAWVNATK-VGKOASNMSA-L", # (3H)Phosphoric acid; Fe +2 ion; Tin, dibutylbis(2,4-pentanedionato-.kappa.O2,.kappa.O4)-, (OC-6-11)-
    "YVBGRQLITPHVOP-UHFFFAOYSA-L", "OPJWPPVYCOPDCM-UHFFFAOYSA-N", "UGTABSAUSFGUPH-UHFFFAOYSA-N", # Triphosphoric acid, sodium salt; Fatty acids, C16-18, 2-ethylhexyl esters; Fatty acids, C8-10, propylene esters
    "ZYBWTEQKHIADDQ-UHFFFAOYSA-N", "RSEKYLFCZNOVCI-UHFFFAOYSA-N", "KSBAEPSJVUENNK-UHFFFAOYSA-L", # Ethanol, mixt. with methanol; Lemon, ext.; Hexanoic acid, 2-ethyl-, tin(2+) salt
    "FXAZUIKRKXGTRK-UHFFFAOYSA-N", "YNPNZTXNASCQKK-LHNTUAQVSA-N", "YXFVVABEGXRONW-JGUCLWPXSA-N", # Disodium methanediylidenebis(azanide); Phenanthrene-d10; Toluene-d8
    "LRHPLDYGYMQRHN-NWURLDAXSA-N", "ISWSIDIOOBJBQZ-QNKSCLMFSA-N", "LQNUZADURLCDLV-RALIUCGRSA-N", # 1-Butanol-d10; Phenol-d6; Nitrobenzene-d5
    "OKKJLVBELUTLKV-MZCSYVLQSA-N", "WDECIBYCCFPHNR-AQZSQYOVSA-N", "UHOVQNZJYSORNB-MZWXYZOWSA-N", # Methan-d3-ol-d; Chrysene-d12; Benzene-d6
    "CWRYPZZKDGJXCA-WHUVPORUSA-N", "RSWGJHLUYNHPMX-UHFFFAOYSA-N", "NIXOWILDQLNWCW-UHFFFAOYSA-M", # Acenaphthene-d10; Tall-oil rosin; 2-Propenoic acid, ion(-1)
    "IWBUOAHJGJZERP-UHFFFAOYSA-N", "VBJGJHBYWREJQD-UHFFFAOYSA-M", "DJEHXEMURTVAOE-UHFFFAOYSA-M" # 2-Propenoic acid, butyl ester, polymer with diethenylbenzene and ethenylbenzene; Sodium phosphate, monobasic, dihydrate; Sulfurous acid, potassium salt (1:1)
]

# Get all unique InChIKeys before filtering
all_unique_inchikey = df_inchikey["InChIKey"].dropna().unique()

# Identify which unique keys will be dropped
dropped_inchikey = pd.Series(all_unique_inchikey)[
    pd.Series(all_unique_inchikey).isin(drop_inchikeys)
].tolist()

# Print
print("Dropped unique InChIKeys:")
for inchi in sorted(dropped_inchikey):
    print(inchi)
print(f"\nDropped {len(dropped_inchikey):,} unique compounds by InChIKey.\n")

# Drop rows from SPECIATE where InChIKey matches any in the list
df_inchikey = df_inchikey[~df_inchikey["InChIKey"].isin(drop_inchikeys)]

# Diagnostic print
print(f"Dropped {len(drop_inchikeys):,} compounds by InChIKey. \n\nRemaining rows: \n{len(df_inchikey):,}\n")


# Define the unwanted chemicals regex (e.g., mixtures, inorganic compounds & inorganic mixtures, etc.)
unwanted_pattern = ("mica|talc|kerosine|kerosene|gasoline|aluminum|aluminium|alumina|barium|beryl|bismuth|"
                    "cadmium|calcium|cesium|caesium|cerium|ceric|chromium|cobalt|copper|cupric|cuprat|cupferron|"
                    "dysprosium|erbium|europium|gallium|gadolin|german|gold|hafnium|holmium|iron|indium|iridium|"
                    "lanthan|lead|lithium|lutetium|magnes|mangan|mercur|molybd|neodym|nickel|niobium|osmium|"
                    "pallad|praseodym|platin|rhodium|rhenium|rhenate|radon|radium|ruthenium|rubidium|"
                    "samarium|scandium|silver|strontium|titan|thorium|thallium|thallic|tantal|tellur|terbium|tungsten|tungstate|"
                    "tin fluoride|vanad|uranium|uranyl|yttrium|ytterbium|zinc|zircon|"
                    "diamond|ammonium tetraborate|ammonium hydrogen phosphate|glycosides|uprous oxide|cryolite|arsenate|"
                    "ferric|emery|ferrous ammonium sulfate|ferrosilicon|dibasic sodium phosphate|"
                    "disodium phosphate dihydrate|disodium phosphate dodecahydrate|monosodium phosphate|sodium phosphate P 32|"
                    "trisodium phosphate|ammonium hydrogen sodium phosphate tetrahydrate|"
                    "foscarnet trisodium|fluorphlogopite|fluoboric acid|ferrous oxide|ferrous sulfide|ferrous chloride|"
                    "phosphate buffered saline|dichloromethane|diborane|boric acid|cristobalite|tritium|naphthalene d8|"
                    "deuterochloroform|deuterium|acid phosphate|aloe vera|air|activated charcoal|actinolite|"
                    "ammonium carbonate|ammonium carbamate|ammonium bisulfite|ammonium bisulfate|"
                    "ammonium ferrous sulfate hexahydrate|dichromate|chromate|"
                    "ammonium phosphate|ammonium perchlorate|ammonium isothiocyanate|"
                    "ammonium iodate|ammonium hydrogen difluoride|ammonium hydrogen carbonate|"
                    "ammonium sulfite|ammonium sulfide|ammonium sulfamate|ammonium sodium hydrogenorthophosphate|"
                    "argon|antimony|antigorite|anhydrite|andalusite|anatase|amorphous silica|ammonium thiosulfate|"
                    "borax|borat|boran|spodumene|bauxite|bakelite|azanide|aventurine|attapulgite|arsine|arsenous|arsenic|"
                    "carbene|calcite|cacodylic acid|alkanes|alkenes|bromoform|iodoform|ftoroform|boron|boric|"
                    "carbon tetrafluoride|carbon tetrachloride|carbon monoxide|carbon disulfide|carbon dioxide|carbon black|"
                    "chlorine|charcoal|cellulase|cellulose|carbonyl sulfide|carbonic acid sodium salt|carbonic acid disodium salt|"
                    "cornmint oil|coal|chrysotile|chromite|chromic|chromate|chloromethane|chlorite|"
                    "alloy|plaster of paris|syenite|natural rubber|mullite|montmorillonite|mineral spirits|radical|"             
                    "potassium phosphate|potassium peroxide|aluminum phosphate|limnanthes alba|"
                    "lewisite|lazurite|langbeinite|kyanite|krypton|kaolinite|iodic acid|illite|yeast|"
                    "hydroxylamine sulfate|hydroxylamine hydrochloride|hydrotalcite|"
                    "hydrogen sulfide|hydrogen selenide|hydrogen peroxide|hydrogen iodide|hydrogen fluoride|hydrogen cyanide|"
                    "hydrochloric acid|hydrobromic acid|huntite|hematite|"
                    "helium|hectorite|gypsum|graphite|grape seed oil|gram's iodine|goethite|fuming sulfuric acid|"
                    "periodic acid|perchloric acid|perboric acid|pentasodium triphosphate|ozone|oxygen|hydroxylapatite|"
                    "orange terpenes|olivine|nitrous oxide|nitrogen dioxide|nitric oxide|nitric acid|"
                    "perlite|zeolite|xenon|wollastonite|vitis vinifera|vermiculite|stainless steel|arsine|"
                    "dolomite|sodium stannate|sodium phosphite|sodium octaborate|sodium peroxydicarbonate|"
                    "sodium monofluorophosphate|sodium methanediphosphonate|sulfur dioxide|borate|"
                    "sodium cyanodithioimidocarbonate|sodium arsenate|sodium methanearsonate|"
                    "tetrasodium pyrophosphate|terpenes|superoxide|starch|stannous|stannane|"
                    "sodium sulfide|sodium sulfate|sodium sulfanilate|sodium silicate|sodium pyrophosphate|"
                    "sodium potassium aluminum silicate|sodium polyphosphate|sodium polymetaphosphate|"
                    "sodium phosphinate|sodium persulfate|pertechnetate|perrhenate|"
                    "peroxoborate|sodium peroxide|sodium permanganate|sodium periodate|sodium perchlorate|"
                    "perborate|sodium oxide|vanadate|nitroprusside|sodium nitrite|sodium nitrate|fullerite|"
                    "monohydrogen phosphate heptahydrate|molybdate|sodium metaphosphate|metaborate|"
                    "sodium metabisulfite|sodium magnesium silicate|sodium iodide|sodium iodate|magnesate|"
                    "sodium hypochlorite|sodium hydrogen sulfate|sodium hydrogen difluoride|"
                    "sodium hydride|sodium hexametaphosphate|sodium hexafluorosilicate|sodium formate|sodium fluoride|"
                    "sodium ferrocyanide|sodium dichromate|sodium chromate|sodium cyanide|sodium cobaltinitrite|"
                    "sodium chlorite|sodium chloride|sodium chlorate|sodium bisulfite|sodium bisulfate|"
                    "sodium bicarbonate|sodium azide|sodium arsenite|sodium aluminosilicate|sodium aluminate|"
                    "potassium tungstate|potassium tripolyphosphate|potassium trifluorido|potassium titanate|"
                    "potassium tin|potassium thiocyanate|potassium tetrafluoroborate|potassium superoxide|"
                    "potassium sulfite|potassium sulfate|potassium pyrosulfate|potassium pyrophosphate|"
                    "potassium polysulfide|potassium phosphate|potassium persulfate|potassium peroxymonosulfate|"
                    "potassium permanganate|potassium periodate|potassium perchlorate|potassium perfluorooctanesulfonate|"
                    "potassium pentaborate tetrahydrate|potassium oxide|potassium nitrite|potassium nitrate|"
                    "potassium metaphosphate|potassium metaborate|potassium metabisulfite|potassium iodide|"
                    "potassium iodate|potassium hydroxide|potassium hydrogen monopersulfate|potassium hydrogen diiodate|"
                    "potassium hydride|potassium hexafluorophosphate|potassium hexacyanoferrate|potassium fluoride|"
                    "potassium ferricyanide|potassium dichromate|potassium bichromate|potassium cyanide|"
                    "potassium chromate|potassium chloride|potassium chlorate|potassium carbonate|potassium bromide|"
                    "potassium bromate|potassium borate|potassium bisulfate|potassium bicarbonate|"
                    "potassium arsenate|phosphorus oxide|phosphorus trifluoride|phosphotungstic acid|phosphorus tribromide|"
                    "phosphorus pentachloride|phosphorus pentasulfide|phosphorous trichloride|phosphorous acid|"
                    "phosphoric trichloride|phosphomolybdic acid|phosphinic acid|metaphosphoric acid|"
                    "silica|silyl|siloxane|silicon|silicic acid|silicate|silan|silicic|selenium|selenious|selenic|"
                    "extract|sapphire|sand|rose oil|aluminosilicate|diopside|quartz|pyrophyllite|borane|"
                    "almandite and pyrope garnet|aluplex|americium 241|ammonium cyanide|amosite|amylase|azurite|"
                    "boehmite|bovine insulin|cerous chloride heptahydrate|chlorous acid|clinoptilolite|"
                    "collagens|cyclic amp|epoxy resin|kinase|lactoferrin|lignin|limonite|"
                    "ferrate|ferrocene|ferrocyanide|ferroin|ferrous gluconate|ferrous lactate|ferrous oxalate|"
                    "ginseng|human chorionic gonadotropin|human insulin|hydrocarbon oils|invert sugar|"
                    "iodine chloride|iodine pentafluoride|iodine pentoxide|iodohippurate sodium|"
                    "15n acid solution|nitrogen mustard hydrochloride|nitrogen tetroxide|nitrogen trifluoride|"
                    "pectin|peroxidase|phenoxy resin|phosphorodithioic acid|polyamide fibers|53249257|"
                    "pyrite|riebeckite|saponins|sepiolite|smithsonite|sodium bromate|sodium bromide|sodium selenate|"
                    "sodium selenite|sodium sulfamate|sodium thiocyanate|sodium thiosulfate|"
                    "sodium trimetaphosphate|sterols|sulfurous acid|sulfuryl chloride|sulfuryl fluoride|terpene resin|"
                    "tetraphenyl tin|thulium|thulium oxide|tungstic acid|uraninite|phytosterols|"
                    "tocopherols|topaz|tremolite|tridymite|triethyltin bromide|triphosphate|triphosphoric acid|"
                    "toximul 3406f|pyrolyzed organic carbon|lanthanum|"
                   )

# Original unwanted pattern string
raw_pattern = unwanted_pattern

# Split into individual terms
terms = raw_pattern.split("|")

# Apply safer regex construction:
# - short/generic tokens → word bounded
# - multi-word phrases → keep as is
safe_terms = []
for t in terms:
    t = t.strip()
    if not t:
        continue
    # heuristic: single word without space → bound it
    if " " not in t:
        safe_terms.append(rf"\b{t}\b")
    else:
        safe_terms.append(t)

# Rebuild regex
safe_pattern = "|".join(safe_terms)

# Get all unique chemical names before filtering
all_unique_chems = df_inchikey["Feature name"].dropna().unique()

# Identify unique chemical names to drop
dropped_chems = [chem for chem in all_unique_chems if pd.Series(chem).str.contains(safe_pattern, case=False, regex=True).any()]

# Print unique chemical names to drop
print(f"Number of dropped unique compounds by keyword: \n{len(dropped_chems):,}\n")
print("Dropped unique compound names:")
for chem in sorted(dropped_chems):
    print(chem)

# Drop mixtures, inorganic compounds & inorganic mixtures, etc. by name/keyword
df_inchikey = df_inchikey[~df_inchikey["Feature name"].str.contains(safe_pattern, case=False, na=False)]

print(f"\nFiltered SPECIATE: \n{initial_n:,} → {len(df_inchikey):,} rows\n")

print("Done keyword-filtering SPECIATE.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")


In [ ]:
# Write keyword-filtered SPECIATE to Excel file
print("Sorting keyword-filtered SPECIATE.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

# Sort
df_inchikey = df_inchikey.sort_values("Feature name")
df_inchikey = df_inchikey.sort_values("weight_fraction")
df_inchikey = df_inchikey.sort_values("PROFILE_CODE")
df_inchikey = df_inchikey.sort_values("CATEGORY_LEVEL_1_Generation_Mechanism")

print("Writing keyword-filtered SPECIATE to Excel.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

# Prevent Excel CAS# conversion
df_inchikey["CAS#"] = df_inchikey["CAS#"].astype(str)
# Save SPECIATE AFTER the InChIKey & name/keyword filtration
df_inchikey.to_excel(r"C:\Users\CountDracula\SPECIATE\SPECIATE_Profiles_All_05_Fltrd.xlsx", index=False)

print("Keyword-filtered SPECIATE written to Excel.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")
print(f"Keyword-filtered SPECIATE Excel shape: \n{df_inchikey.shape[0]:,} rows × {df_inchikey.shape[1]} columns\n")


In [ ]:
# QC print
print(f"Keyword-filtered SPECIATE shape: \n{df_inchikey.shape[0]:,} rows × {df_inchikey.shape[1]} columns\n")

print(f"Keyword-filtered SPECIATE columns:")
for col in df_inchikey.columns:
    print(col)
print()


In [ ]:
# Statistics accumulators
print("Creating statistics accumulators.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

df_stat = df_inchikey

total_rows = len(df_stat)

unique_profiles = set()
unique_orig_names = set()
unique_feat_names = set()

casrn_profile_stats = defaultdict(lambda: {
    "Feature name": None,
    "row_count": 0,
    "profiles": set(),
    "weight_fraction": []
})

column_null_counts = {}
overall_values_per_column = {}
unique_values_per_column = {}

value_counts_limited = {}  # optional sample column

print("Statistics accumulators created.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")


In [ ]:
# Column-level statistics
print("Calculating column-level statistics.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

for col in df_stat.columns:
    non_null = df_stat[col].dropna()

    unique_values_per_column[col] = set(non_null.unique())
    overall_values_per_column[col] = non_null.shape[0]
    column_null_counts[col] = df_stat[col].isna().sum()

print("Column-level statistics calculated.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")


In [ ]:
# Global unique entities
print("Calculating global unique entities.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

unique_profiles.update(df_stat["PROFILE_NAME"].dropna().unique())
unique_orig_names.update(df_stat["Name"].dropna().unique())
unique_feat_names.update(df_stat["Feature name"].dropna().unique())

print("Global unique entities calculated.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")


In [ ]:
# CAS# → profile/weight fraction relationships
print("Establishing CAS# & profile/weight fraction relationships.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

for _, row in df_stat.iterrows():
    casrn = row.get("CAS#")

    if pd.isna(casrn):
        continue

    s = casrn_profile_stats[casrn]

    if "weight_fraction" not in s:
        s["weight_fraction"] = []
    if "profiles" not in s:
        s["profiles"] = set()
    if "row_count" not in s:
        s["row_count"] = 0
    if "Feature name" not in s:
        s["Feature name"] = None

    # Row-level count
    s["row_count"] += 1

    # Chemical name (store once)
    chem_name = row.get("Feature name")
    if pd.notna(chem_name) and s["Feature name"] is None:
        s["Feature name"] = chem_name

    # Profile names
    profile = row.get("PROFILE_NAME")
    if pd.notna(profile):
        s["profiles"].add(profile)

    # Weight fractions
    cwf = row.get("weight_fraction")
    if pd.notna(cwf):
        s["weight_fraction"].append(cwf)

print("CAS# & profile/weight fraction relationships established.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")


In [ ]:
# Summary
print("===== OVERALL & UNIQUE VALUE COUNTS PER COLUMN =====\n")

rows = []

for col in unique_values_per_column:
    rows.append({
        "Column": col,
        "Overall": overall_values_per_column.get(col, 0),
        "Unique": len(unique_values_per_column[col]),
        "Total rows": total_rows
    })

summary_df = pd.DataFrame(rows)

summary_df.style.format({
    "Overall": "{:,}",
    "Unique": "{:,}",
    "Total rows": "{:,}"
}).set_table_styles([
    {"selector": "th, td", "props": [("border", "1px solid black"), ("text-align", "left")]},
    {"selector": "table", "props": [("border-collapse", "collapse")]}
])


In [ ]:
# Show top N most frequent compounds
TOP_N = 30

# Sort CAS# by row_count descending
sorted_casrn = sorted(
    casrn_profile_stats.items(),
    key=lambda x: x[1]["row_count"],
    reverse=True
)

# Build table rows
rows = []

for casrn, stats in sorted_casrn[:TOP_N]:
    chem_name = stats["Feature name"] or "Unknown"
    row_count = stats["row_count"]
    unique_profiles = len(stats["profiles"])

    # Wight fraction stats
    if stats["weight_fraction"]:
        cwf_min = np.min(stats["weight_fraction"])
        cwf_max = np.max(stats["weight_fraction"])
        cwf_mean = np.mean(stats["weight_fraction"])
    else:
        cwf_min = cwf_max = cwf_mean = np.nan

    rows.append({
        "CAS#": casrn,
        "Chemical name": chem_name,
        "Occurrences (rows)": row_count,
        "Unique profiles": unique_profiles,
        "WF min": cwf_min,
        "WF max": cwf_max,
        "WF mean": cwf_mean
    })

# Create DataFrame
casrn_df = pd.DataFrame(rows)

# Render nicely with commas and borders
casrn_df.style.format({
    "Occurrences (rows)": "{:,}",
    "Unique profiles": "{:,}",
    "Central WF min": "{:.5g}",
    "Central WF max": "{:.5g}",
    "Central WF mean": "{:.5g}"
}).set_table_styles([
    {"selector": "th, td", "props": [("border", "1px solid black"), ("text-align", "left")]},
    {"selector": "table", "props": [("border-collapse", "collapse")]}
])


In [ ]:
# Write stats DF to Excel file
print("Writing stats DF to Excel.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

rows = []
for casrn, s in casrn_profile_stats.items():
    rows.append({
        "CAS#": casrn,
        "Feature name": s["Feature name"] or "Unknown",
        "Row count": s["row_count"],
        "Unique profiles": len(s["profiles"]),
        "WF mean": np.mean(s["weight_fraction"]) if s["weight_fraction"] else np.nan,
        "WF min": min(s["weight_fraction"]) if s["weight_fraction"] else np.nan,
        "WF max": max(s["weight_fraction"]) if s["weight_fraction"] else np.nan,
    })

pd.DataFrame(rows).to_excel(r"C:\Users\CountDracula\SPECIATE\SPECIATE_Profiles_All_05_CAS_Profile_Composition.xlsx", index=False)

print("Stats DF written to Excel.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")
